# The normalization test (Study 1 extension)

Which normalization rule the replication, and Studies 2 and 3, use (gp_reward-priors
`HANDOFF_HP_SELECTION.md` §14.104, amended by §14.108 and §14.110; built in §14.109). Four arms on
Study 1's reward models 0-10, with new IQL seeds 50-60 and leak-free trajectory cuts
(`return_range_split "length"`):

- **S**: Study 1's selection process, re-run leak-free (its seed-50 grid winner);
- **N**: index 0 (the raw reward);
- **F**: index 2;
- **O3**: one index per dataset shared by both protocols, run only if N and F both fail.

The decision is registered in `results/normtest_readout.py`: one rule must pass under BOTH
protocols, tried in the order N, F, O3, else S.

**Gate (§14.104's do-not):** nothing of an arm's evaluation cells is shown until every cell the
decision reads is complete. Until then this notebook shows run states only. S's seed-50 grids are
drawn once all 32 are complete, since their winners must be written before S's cells exist.

Descriptive figures compare each family only with itself (arm against arm), never one family
with another.

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb

HERE = os.path.abspath(".")
ROOT = os.path.dirname(HERE)
sys.path.insert(0, HERE)
sys.path.insert(0, ROOT)
import eval_readout as R  # noqa: E402
import iql_score  # noqa: E402
import normtest_readout as NR  # noqa: E402  the registered decision

_argv, sys.argv = sys.argv, [sys.argv[0]]
import normtest as NT  # noqa: E402
import phase2_sweeps as P  # noqa: E402

sys.argv = _argv

SCALE = 100.0
api = wandb.Api(timeout=120)
GWIN_PATH = os.path.join(ROOT, NT.WINNERS)
gwin = json.load(open(GWIN_PATH)) if os.path.exists(GWIN_PATH) else {}
FAMS = list(NT.ANCHORS)  # mr, pt, nmr, ens_mean
PROT_LABEL = {"std": "with terminals", "cand": "without terminals"}

## Loading

The evaluation cells and zero cells through the registered readout's own collector
(`normtest_readout.collect`). S's grids (seed 50) are found the way `normtest.py winners` finds
them.

In [ ]:
cells, zeros, notes = NR.collect(api, P, NT)
for n in notes:
    print("note:", n)

sp = P.spec()
grid_fixed = {
    NT.grid_name(v, f, p): NT.family_fixed(sp, f, v, p)
    for v in P.VARIANTS
    for p in NT.PROTOCOLS
    for f in FAMS
}
grid_sweeps = {}
for sw in P.iql_sweeps(api):
    params = sw.config.get("parameters")
    hits = P.match_entry(params, grid_fixed)
    flat = P.flat(params)
    if (
        len(hits) == 1
        and flat.get("seed") == NT.GRID_SEED
        and flat.get("normalize_reward") == list(P.STAGE4_INDICES)
    ):
        grid_sweeps.setdefault(hits[0], []).append(sw)


def grid_state(name):
    sws = grid_sweeps.get(name, [])
    if not sws:
        return "no sweep"
    if len(sws) > 1:
        return f"{len(sws)} sweeps"
    runs = list(sws[0].runs)
    return f"{sum(r.state == 'finished' for r in runs)}/8"


grids_done = sum(grid_state(n) == "8/8" for n in grid_fixed)
print(f"S's grids complete: {grids_done}/{len(grid_fixed)}; winners written: "
      f"{sum(n in gwin for n in grid_fixed)}/{len(grid_fixed)}")

## Progress (states only)

One row per family, dataset and protocol: S's grid (finished runs of 8), then each evaluation
cell's state. S's cell exists only once its grid winner is written (stage 2). No score is
shown.

In [ ]:
def cell_state(c):
    why = R.cell_ready(c)
    return why if why else "complete"


rows = []
for p in NT.PROTOCOLS:
    for v in P.VARIANTS:
        for f in FAMS:
            g = NT.grid_name(v, f, p)
            s_idx = gwin.get(g)
            rows.append({
                "protocol": PROT_LABEL[p], "dataset": v, "family": NT.METHOD[f],
                "S grid (seed 50)": grid_state(g),
                "S winner": "" if s_idx is None else s_idx,
                "S cell": "awaits its grid" if s_idx is None else cell_state(cells.get((f, v, p, s_idx))),
                "N (index 0)": cell_state(cells.get((f, v, p, 0))),
                "F (index 2)": cell_state(cells.get((f, v, p, 2))),
            })
progress = pd.DataFrame(rows).set_index(["protocol", "dataset", "family"])
zero_rows = [{"protocol": PROT_LABEL[p], "dataset": v, "zero cell": cell_state(zeros.get((v, p)))}
             for p in NT.PROTOCOLS for v in P.VARIANTS]
display(progress)
display(pd.DataFrame(zero_rows).set_index(["protocol", "dataset"]))

## S's selection grids (seed 50), once all 32 are complete

Each family's seed-50 score at every index 0-7, for each dataset and protocol: Study 1's stage-4
rule on reward model 0 picks S's index (marked ★). Indices 0 (N) and 2 (F) are shaded. One panel
per family and dataset: each panel compares one reward with itself across indices.

In [ ]:
def grid_scores(name):
    sw = grid_sweeps[name][0]
    runs = [iql_score.score_run(r) for r in sw.runs]
    win, table = iql_score.select_normalization(runs, list(P.STAGE4_INDICES))
    return int(win), {t["normalize_reward"]: t["score"] for t in table}


if grids_done < len(grid_fixed):
    print(f"Not drawn: {grids_done} of {len(grid_fixed)} grids complete.")
else:
    for p in NT.PROTOCOLS:
        fig, axes = plt.subplots(len(P.VARIANTS), len(FAMS), figsize=(14, 10), sharex=True)
        for i, v in enumerate(P.VARIANTS):
            for j, f in enumerate(FAMS):
                ax = axes[i, j]
                win, sc = grid_scores(NT.grid_name(v, f, p))
                idx = sorted(sc)
                ys = [SCALE * sc[k] for k in idx]
                ax.axvspan(-0.4, 0.4, color="0.85", zorder=0)
                ax.axvspan(1.6, 2.4, color="0.85", zorder=0)
                ax.plot(idx, ys, "o-", color="C0", lw=1.2, ms=4)
                ax.plot([win], [SCALE * sc[win]], "*", color="C3", ms=13, zorder=3)
                ax.set_xticks(idx)
                ax.grid(alpha=0.3)
                if i == 0:
                    ax.set_title(NT.METHOD[f])
                if j == 0:
                    ax.set_ylabel(f"{v}\nscore × 100")
                if i == len(P.VARIANTS) - 1:
                    ax.set_xlabel("normalization index")
        fig.suptitle(f"S's seed-50 grids, {PROT_LABEL[p]} (★ = S's index; shaded = N's and F's)")
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        plt.show()

## The registered decision

`normtest_readout.run`, unchanged: it prints NOT READ, with the cells still missing, until every
cell is complete.

In [ ]:
lines = []
rule = NR.run(cells, zeros, gwin, NT, P.VARIANTS, out=lines.append)
print("\n".join(lines))
readable = not any("NOT READ" in l for l in lines)

## Each arm against S, per family (descriptive, after the decision is readable)

Test (i)'s quantity: S_A − S_S per family, pooled over the four datasets with its exact stratified
95% interval, for each alternative arm and protocol. The dashed line is the −0.10 margin. Each
point is one family's own difference.

In [ ]:
if not readable:
    print("Not drawn: the decision is not readable yet (§14.104).")
else:
    fams = [NT.METHOD[f] for f in FAMS]
    arms = [a for a in NR.ARMS if a in NT.ARM_INDEX or all(NT.o3_key(v) in gwin for v in P.VARIANTS)]
    fig, axes = plt.subplots(1, len(NT.PROTOCOLS), figsize=(13, 4.5), sharey=True)
    for ax, p in zip(axes, NT.PROTOCOLS):
        S = {(NT.METHOD[f], v): cells[(f, v, p, gwin[NT.grid_name(v, f, p)])] for f in FAMS for v in P.VARIANTS}
        Z = {v: zeros[(v, p)] for v in P.VARIANTS}
        for k, arm in enumerate(arms):
            idx = {v: NT.ARM_INDEX[arm] if arm in NT.ARM_INDEX else gwin[NT.o3_key(v)] for v in P.VARIANTS}
            if not all((f, v, p, idx[v]) in cells for f in FAMS for v in P.VARIANTS):
                continue
            A = {(NT.METHOD[f], v): cells[(f, v, p, idx[v])] for f in FAMS for v in P.VARIANTS}
            j = NR.judge(A, S, Z, fams, P.VARIANTS)
            for i, f in enumerate(fams):
                e = j["i"][f]["e"]
                x = i + (k - (len(arms) - 1) / 2) * 0.22
                ax.plot([x, x], [e["lower"], e["upper"]], color=f"C{k}", lw=3)
                ax.plot(x, e["mean"], "o", color=f"C{k}", label=arm if i == 0 else None)
        ax.axhline(0, color="k", lw=0.8)
        ax.axhline(-NR.MARGIN, color="k", ls="--", lw=1)
        ax.set_xticks(range(len(fams)))
        ax.set_xticklabels(fams)
        ax.set_title(PROT_LABEL[p])
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("S_A − S_S (last-10, pooled; 95% exact)")
    axes[0].legend(frameon=False)
    fig.suptitle("Each alternative arm against S, within each family")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()